In [9]:
import os
import json
import urllib.request
import urllib.error
import openai
from dotenv import load_dotenv
load_dotenv()  # .env 파일 자동 로드

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def _get(path: str):
    url = f"{BASE_URL}{path}"
    headers = {
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0 (compatible; AI-Client/1.0)",
    }
    req = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            data = resp.read().decode("utf-8")
        return json.loads(data)
    except urllib.error.HTTPError as e:
        return {"error": f"HTTP {e.code}", "url": url}
    except urllib.error.URLError as e:
        return {"error": f"URL error: {e.reason}", "url": url}


def get_popular_movies():
    return json.dumps(_get("/movies"), ensure_ascii=False)


def get_movie_details(id):
    return json.dumps(_get(f"/movies/{id}"), ensure_ascii=False)


def get_movie_credits(id):
    return json.dumps(_get(f"/movies/{id}/credits"), ensure_ascii=False)


FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


'[{"adult": false, "backdrop_path": "https://image.tmdb.org/t/p/w1280/7HKpc11uQfxnw0Y8tRUYn1fsKqE.jpg", "genre_ids": [878, 28, 53], "id": 1236153, "original_language": "en", "original_title": "Mercy", "overview": "In the near future, a detective stands on trial accused of murdering his wife. He has ninety minutes to prove his innocence to the advanced AI Judge he once championed, before it determines his fate.", "popularity": 649.4032, "poster_path": "https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg", "release_date": "2026-01-20", "title": "Mercy", "video": false, "vote_average": 7.054, "vote_count": 468}, {"adult": false, "backdrop_path": "https://image.tmdb.org/t/p/w1280/hHDNOlATHhre4eZ7aYz5cdyJLik.jpg", "genre_ids": [27, 53, 878], "id": 1272837, "original_language": "en", "original_title": "28 Years Later: The Bone Temple", "overview": "Dr. Kelson finds himself in a shocking new relationship - with consequences that could change the world as they know it - and Spike\'

In [10]:
SYSTEM_PROMPT = """
너는 Movie Expert Agent이다.
사용자의 질문을 이해하고 아래 함수 중 하나를 선택해 호출하라.
필요할 때만 함수를 호출하고, 함수명과 인자를 정확히 지정하라.

사용 가능한 함수:
- get_popular_movies(): /movies에서 인기 영화 목록을 가져온다.
- get_movie_details(id): /movies/:id에서 영화 정보를 가져온다.
- get_movie_credits(id): /movies/:id/credits에서 출연진 및 제작진을 가져온다.
"""

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "인기 영화 목록을 가져온다.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "영화 ID로 영화 상세 정보를 가져온다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "영화의 ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "영화 ID로 출연진 및 제작진 정보를 가져온다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "영화의 ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

messages = [{"role": "system", "content": SYSTEM_PROMPT}]


In [11]:
from openai.types.chat import ChatCompletionMessage


def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments) if arguments else {}
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)
            result = function_to_run(**arguments)

            print(f"Ran {function_name} with args {arguments}")

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)


In [12]:
tests = [
    "지금 인기 있는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘",
]

for t in tests:
    messages.append({"role": "user", "content": t})
    print(f"\nUser: {t}")
    call_ai()



User: 지금 인기 있는 영화가 무엇인지 알려줘
Calling function: get_popular_movies with {}
Ran get_popular_movies with args {}
AI: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Mercy**
   - 개봉일: 2026-01-20
   - 평점: 7.0
   - 줄거리: 미래의 한 형사가 아내를 살해한 혐의로 기소되어 AI 판사에게 무죄를 입증해야 하는 이야기입니다.
   - 포스터:
     ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **28 Years Later: The Bone Temple**
   - 개봉일: 2026-01-14
   - 평점: 7.2
   - 줄거리: 드.켈슨은 그의 세계를 바꿀 수 있는 충격적인 관계에 놓이게 됩니다.
   - 포스터:
     ![28 Years Later: The Bone Temple](https://image.tmdb.org/t/p/w780/kK1BGkG3KAvWB0WMV1DfOx9yTMZ.jpg)

3. **Les Orphelins**
   - 개봉일: 2025-08-20
   - 평점: 6.0
   - 줄거리: 두 친구가 첫 사랑의 죽음 이후 함께 사건을 해결하기 위한 여정을 시작하는 이야기입니다.
   - 포스터:
     ![Les Orphelins](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **A Woman Scorned**
   - 개봉일: 2025-06-09
   - 평점: 6.0
   - 줄거리: 자매가 한 그룹 남자들에 의해 공격당한 후 복수를 결심하는 이야기입니다.
   - 포스터:
     ![A Woman Scorned](https://image.tmdb.org/t/p/w780/dlOSBiNULMPzKIze84LDjvEN9z1.jpg)